# EventFlow — Person A

Run the commands in the repository README first. This notebook is a walkthrough of the same pipeline and shows the artifacts for the classical analysis.

In [ ]:
from pathlib import Path
import sys
import subprocess
import pandas as pd
from IPython.display import display

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
results = root / "results"
assert (root / "pyproject.toml").is_file(), "Open this notebook from the repo root or notebooks folder."


## Run the pipeline

Set `rerun` to True to recompute validation experiments from the CSVs in `data/raw/`.

In [ ]:
rerun = False
if rerun:
    subprocess.run([sys.executable, "-m", "eventflow.run", "develop"], cwd=root, check=True)
    subprocess.run([sys.executable, "scripts/report_assets.py"], cwd=root, check=True)


## Data checks

Which classes dominate? Which sensor fields are missing? Compare variation between readings with variation over the whole dataset. These summaries do not by themselves identify sensor noise.

In [ ]:
for name in ["event_balance", "time_gaps", "missingness", "sensor_variability"]:
    display(pd.read_csv(results / "eda" / f"{name}.csv"))


## Model comparison and ablations

Positive change in log loss indicates the removed group helped in this setup.

In [ ]:
runs = pd.read_csv(results / "latest_experiments.csv")
display(runs[["run", "validation_log_loss"]].sort_values("validation_log_loss"))
display(pd.read_csv(results / "ablations.csv"))


## Validation errors

Compare the class and prefix-length buckets with their sample sizes. Look at the probability assigned to the true class in the highest-loss cases.

In [ ]:
for name in ["errors_by_class", "errors_by_length", "highest_loss_examples", "confidence_bins"]:
    display(pd.read_csv(results / f"{name}.csv").head(14))
